In [ ]:
import numpy as np
from scipy.optimize import minimize

# Data for dishes: [Price(IDR), Cost_per_g, S_base, D_base, R_i, Alpha]
# S_base, D_base, R_i comes from data
C_budget = 25000
dishes = {
    '31_Orek': [4000, 15, 29.59, 22, 0.61, 0.10], 
    '23_BaliTahu': [4000, 12, 34.64, 22, 0.47, 0.8],
    '15_TelurMata': [6000, 45, 47.75, 20, 0.75, 0.8], 
    '13_Ayambistik': [15000, 75, 55.32, 22, 0.67, 1.6],
    '12_Ayamlaos': [15000, 85, 52.25, 16, 0.30, 1.6], 
    '11_Rendang': [18000, 95, 54.39, 18, 0.54, 1.6],
    '9_BaliTelur': [7000, 48, 56.10, 21, 0.73, 0.8], 
    '7_Ikan': [16000, 80, 39.00, 20, 0.53, 1.6],
    '1_Nasi': [5000, 6, 146.32, 78, 0.62, 0.10]
}


def objective(S):
    total_profit = 0
    for idx, (name, data) in enumerate(dishes.items()):
        P, C, S_base, D_base, R, alpha = data
        # Demand function
        Q = D_base * (S[idx] / S_base)**alpha
        profit = (P - C * S[idx]) * Q
        total_profit -= profit  
    return total_profit

# Constraints
cons = []
for idx, (name, data) in enumerate(dishes.items()):
    lower_bound = data[2] * data[4] * 1.15 # safety buffer
    upper_bound = data[2] 
    # Lower Bound - Consumption Constraints
    cons.append({'type': 'ineq', 'fun': lambda S, lb=lower_bound, i=idx: S[i] - lb})
    # Upper Bound - Base Serving Size Constraints
    cons.append({'type': 'ineq', 'fun': lambda S, ub=upper_bound, i=idx: ub - S[i]})

# Budget Constraint:
def budget_con(S):
    total_cost = 0
    for idx, (name, data) in enumerate(dishes.items()):
        C, S_base, D_base, alpha = data[1], data[2], data[3], data[5]
        Q = D_base * (S[idx] / S_base)**alpha
        total_cost += C * S[idx] * Q
    return C_budget - total_cost 

cons.append({'type': 'ineq', 'fun': budget_con})

# Initial guess
S0 = [d[2] for d in dishes.values()]
res = minimize(objective, S0, method='SLSQP', constraints=cons)
print(res.x) # Optimized serving sizes

[ 20.96877576  34.64005149  47.75001291  55.32017376  52.2501261
  54.39016816  56.05736366  39.00029657 104.32615375]
